# M5: обучение CatBoost и LightGBM на Kaggle

Дневной прогноз продаж всех **30 490 пар «товар — магазин»** на следующие 28 дней. Используем 63 признака и считаем WRMSSE на 12 уровнях M5. Чем меньше WRMSSE, тем лучше.

Здесь сохранены перенос CatBoost, первый LightGBM и проверка более плотной выборки. Предыдущие эксперименты с признаками и Optuna находятся в [третьем ноутбуке](03_kaggle_wrmsse.ipynb).

**Где остановились:** CatBoost пока оставляем на схеме **52 × 2**. Для LightGBM схема **52 × 4** дала WRMSSE **0.55332** против **0.55498** на одном историческом окне. Следующий опыт — **52 × 8 для LightGBM** с замером времени и памяти, затем проверка на других окнах.

Сохранённые выводы показывают уже выполненные запуски. Полное выполнение снова обучит модели LightGBM и обе модели из раздела 7. Флаги `RUN_PUBLIC` и `RUN_CV` управляют только исходными запусками CatBoost в разделе 5.

Результаты рассчитаны внутри ноутбука. Официального score после отправки submission пока нет. Public использовали при разработке; private-период ещё не оценивали.


## 1. Окружение и исходные данные

Ноутбук рассчитан на Kaggle CPU. В сессии использованы CatBoost 1.2.10 и LightGBM 4.6.0. В Input нужно подключить M5 Forecasting — Accuracy.

Первая ячейка находит CSV, создаёт каталоги результатов и задаёт локальные параметры Optuna. `RUN_PUBLIC` и `RUN_CV` сейчас выключены. Сохранённые выводы — история уже выполненных опытов, а не команда снова их обучать.


In [20]:
from pathlib import Path
from functools import lru_cache
from time import perf_counter
import gc
import hashlib
import json
import platform

import catboost
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from IPython.display import display
from IPython.utils.capture import capture_output

RUN_PUBLIC = False
RUN_CV = False
HORIZON = 28
RANDOM_STATE = 42
CB_N_ORIGINS = 52

PROJECT_ROOT = Path("/kaggle/working/demand-planner")
RESULTS = PROJECT_ROOT / "results" / "m5_wrmsse"
MODELS = PROJECT_ROOT / "models"
SCRATCH = Path("/tmp/demand-planner")
for directory in (RESULTS, MODELS, SCRATCH):
    directory.mkdir(parents=True, exist_ok=True)

required_files = (
    "calendar.csv",
    "sales_train_validation.csv",
    "sell_prices.csv",
    "sales_train_evaluation.csv",
)
candidates = sorted(
    {
        path.parent
        for path in Path("/kaggle/input").rglob("calendar.csv")
        if all((path.parent / filename).is_file() for filename in required_files)
    }
)
if len(candidates) != 1:
    raise FileNotFoundError(
        "Подключи M5 Forecasting - Accuracy через Input → Add Input. "
        f"Найдено подходящих папок: {len(candidates)}."
    )
DATA_RAW = candidates[0]

LOCAL_REFERENCE = {
    "trial": 8,
    "params": {
        "loss_function": "Tweedie:variance_power=1.5",
        "eval_metric": "Tweedie:variance_power=1.5",
        "random_seed": 42,
        "thread_count": 4,
        "allow_writing_files": False,
        "depth": 4,
        "learning_rate": 0.03367321864197597,
        "l2_leaf_reg": 1.579816405580968,
        "iterations": 768,
    },
    "features": [
        "item_id",
        "dept_id",
        "cat_id",
        "store_id",
        "state_id",
        "mean_7d",
        "mean_28d",
        "mean_56d",
        "std_28d",
        "zero_share_28d",
        "trend_7d_28d",
        "weekday_mean_4w",
        "lag_28d",
        "lag_35d",
        "lag_42d",
        "lag_56d",
        "lag_364d",
        "horizon",
        "day_of_week",
        "day_of_month",
        "month",
        "week_of_year",
        "year",
        "snap",
        "has_event",
        "price_last_known",
        "price_observed_this_week",
        "price_change_1w",
        "price_change_4w",
        "price_vs_mean_4w",
        "price_vs_mean_13w",
        "price_vs_max_13w",
        "price_relative_std_13w",
        "price_observed_share_13w",
        "days_since_last_sale",
        "has_never_sold",
        "sale_day_share_7d",
        "sale_day_share_56d",
        "sale_frequency_change_7d_vs_prev21d",
        "mean_positive_sale_28d",
        "mean_positive_sale_56d",
        "positive_sale_cv2_56d",
        "max_daily_sale_28d",
        "store_dept_mean_7d",
        "store_dept_mean_28d",
        "store_dept_mean_56d",
        "store_dept_trend_7d_28d",
        "store_dept_trend_28d_56d",
        "store_dept_cv_28d",
        "store_dept_weekday_mean_4w",
        "store_dept_weekday_factor_4w",
        "item_share_store_dept_28d",
        "item_share_change_7d_28d",
        "other_stores_mean_7d",
        "other_stores_mean_28d",
        "other_stores_mean_56d",
        "other_stores_trend_7d_28d",
        "other_stores_trend_28d_56d",
        "other_stores_cv_28d",
        "other_stores_weekday_mean_4w",
        "other_stores_weekday_factor_4w",
        "own_vs_other_stores_mean_28d",
        "own_vs_other_stores_trend",
    ],
    "categorical_features": ["item_id", "dept_id", "cat_id", "store_id", "state_id"],
    "target_scale": 1.0,
    "mean_WRMSSE": 0.6463928135345688,
    "control_mean_WRMSSE": 0.6564511185747867,
    "gain_vs_control": 0.01532224526032544,
    "folds": [
        {
            "origin": "2016-01-31",
            "WRMSSE": 0.7206672668411251,
            "Bias": -0.03188512734307103,
        },
        {
            "origin": "2016-02-28",
            "WRMSSE": 0.6516825949941355,
            "Bias": -0.0026594123740217857,
        },
        {
            "origin": "2016-03-27",
            "WRMSSE": 0.5668285787684458,
            "Bias": -0.016609536359959987,
        },
    ],
    "source": "optuna",
    "public_evaluated": True,
    "pipeline_hash": "bfdd1373c40050099482c934fd3caef08bea0a1b07fadae9052cf0d1b490a0a4",
    "raw_files": {
        "calendar.csv": {"size": 103469, "mtime_ns": 1792003186940192762},
        "sales_train_validation.csv": {
            "size": 120007726,
            "mtime_ns": 1792003187051472135,
        },
        "sell_prices.csv": {"size": 203395785, "mtime_ns": 1792003187537067875},
    },
    "training_origins": 52,
    "horizons_per_origin": 2,
    "train_rows": 3170960,
    "public_origin": "2016-04-24",
    "public_metrics": {"WRMSSE": 0.6367278319446078, "Bias": -0.04198588911800183},
    "model": "models/03_daily_catboost_optuna_v9.cbm",
}
FEATURE_COLUMNS = list(LOCAL_REFERENCE["features"])
CATEGORY_COLUMNS = list(LOCAL_REFERENCE["categorical_features"])
CATBOOST_PARAMS = dict(LOCAL_REFERENCE["params"])
TARGET_SCALE = float(LOCAL_REFERENCE["target_scale"])
price_target_scale = TARGET_SCALE
PUBLIC_ORIGIN = pd.Timestamp(LOCAL_REFERENCE["public_origin"])
cb_public_origin = PUBLIC_ORIGIN
CV_ORIGINS = [pd.Timestamp(row["origin"]) for row in LOCAL_REFERENCE["folds"]]

runtime_versions = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "catboost": catboost.__version__,
}
pd.set_option("display.max_columns", 30)
print("Данные:", DATA_RAW)
print("Версии:", runtime_versions)
print("Обучение public:", RUN_PUBLIC, "| исторические окна:", RUN_CV)
print("Параметры:", CATBOOST_PARAMS)


Данные: /kaggle/input/competitions/m5-forecasting-accuracy
Версии: {'python': '3.13.15', 'numpy': '2.1.3', 'pandas': '2.3.3', 'catboost': '1.2.10'}
Обучение public: True | исторические окна: True
Параметры: {'loss_function': 'Tweedie:variance_power=1.5', 'eval_metric': 'Tweedie:variance_power=1.5', 'random_seed': 42, 'thread_count': 4, 'allow_writing_files': False, 'depth': 4, 'learning_rate': 0.03367321864197597, 'l2_leaf_reg': 1.579816405580968, 'iterations': 768}


### 1.1. Календарь и периоды

Доступная история заканчивается **24 апреля 2016 года**. Public — **25 апреля — 22 мая**, private — **23 мая — 19 июня**. Последний период сохраняем для итоговой проверки.


In [21]:
calendar = pd.read_csv(DATA_RAW / "calendar.csv", parse_dates=["date"])
calendar_by_day = calendar.set_index("d")

periods = pd.DataFrame(
    [
        {"period": "История", "first_day": "d_1", "last_day": "d_1913"},
        {"period": "Public / разработка", "first_day": "d_1914", "last_day": "d_1941"},
        {
            "period": "Private / финальная проверка",
            "first_day": "d_1942",
            "last_day": "d_1969",
        },
    ]
)

periods["start"] = periods["first_day"].map(calendar_by_day["date"])
periods["end"] = periods["last_day"].map(calendar_by_day["date"])
periods["days"] = (periods["end"] - periods["start"]).dt.days + 1

assert periods.loc[periods["period"].ne("История"), "days"].eq(HORIZON).all()
display(periods.set_index("period"))


,first_day,last_day,start,end,days
period,,,,,
История,d_1,d_1913,2011-01-29,2016-04-24,1913
Public / разработка,d_1914,d_1941,2016-04-25,2016-05-22,28
Private / финальная проверка,d_1942,d_1969,2016-05-23,2016-06-19,28


### 1.2. Товары и уровни иерархии

Проверяем состав данных: 3 049 товаров, 10 магазинов, 30 490 исходных рядов. После агрегации по штатам, магазинам, категориям и отделам получаем 42 840 рядов на 12 уровнях.


In [22]:
identifier_columns = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]

series_metadata = pd.read_csv(
    DATA_RAW / "sales_train_validation.csv",
    usecols=identifier_columns,
    dtype="string",
)

assert not series_metadata.duplicated(["item_id", "store_id"]).any()
assert len(series_metadata) == 30_490
assert series_metadata["store_id"].nunique() == 10

M5_LEVELS = {
    "Total": [],
    "State": ["state_id"],
    "Store": ["store_id"],
    "Category": ["cat_id"],
    "Department": ["dept_id"],
    "State / Category": ["state_id", "cat_id"],
    "State / Department": ["state_id", "dept_id"],
    "Store / Category": ["store_id", "cat_id"],
    "Store / Department": ["store_id", "dept_id"],
    "Item": ["item_id"],
    "Item / State": ["item_id", "state_id"],
    "Item / Store": ["item_id", "store_id"],
}

hierarchy = pd.DataFrame(
    [
        {
            "level": level,
            "series": (
                series_metadata.groupby(columns, observed=True).ngroups
                if columns
                else 1
            ),
        }
        for level, columns in M5_LEVELS.items()
    ]
).set_index("level")

assert int(hierarchy["series"].sum()) == 42_840
display(hierarchy)
print(f"Всего оцениваемых рядов: {hierarchy['series'].sum():,}")


,series
level,
Total,1
State,3
Store,10
Category,3
Department,7
State / Category,9
State / Department,21
Store / Category,30
Store / Department,70


Всего оцениваемых рядов: 42,840


### 1.3. Продажи и цены для расчёта весов

Историю читаем из `sales_train_validation.csv`. Для денежных весов берём цены и продажи последних 28 доступных дней. Целевые продажи public в обучающую историю не входят.


In [23]:
M5_KEYS = ["store_id", "item_id"]
TRAIN_END_DAY = 1913

train_day_columns = [f"d_{day}" for day in range(1, TRAIN_END_DAY + 1)]

series_index = pd.MultiIndex.from_frame(series_metadata[M5_KEYS])

train_daily = pd.read_csv(
    DATA_RAW / "sales_train_validation.csv",
    usecols=[*M5_KEYS, *train_day_columns],
    dtype={
        "store_id": "string",
        "item_id": "string",
        **{column: "int32" for column in train_day_columns},
    },
)

train_daily = (
    train_daily.set_index(M5_KEYS).reindex(series_index).loc[:, train_day_columns]
)

train_daily.columns = pd.DatetimeIndex(
    calendar.set_index("d").loc[train_day_columns, "date"],
    name="date",
)

weight_dates = train_daily.columns[-28:]

weight_weeks = calendar.set_index("date").loc[weight_dates, "wm_yr_wk"].unique()

price_parts = []

for chunk in pd.read_csv(
    DATA_RAW / "sell_prices.csv",
    usecols=["store_id", "item_id", "wm_yr_wk", "sell_price"],
    dtype={
        "store_id": "string",
        "item_id": "string",
        "wm_yr_wk": "int32",
        "sell_price": "float64",
    },
    chunksize=200_000,
):
    selected = chunk.loc[chunk["wm_yr_wk"].isin(weight_weeks)]

    if not selected.empty:
        price_parts.append(selected.copy())

prices_for_weights = pd.concat(
    price_parts,
    ignore_index=True,
)

del price_parts

print(f"Рядов: {len(train_daily):,}")
print(f"Дней истории: {train_daily.shape[1]:,}")
print(
    "Период для весов:",
    weight_dates.min().date(),
    "—",
    weight_dates.max().date(),
)


Рядов: 30,490
Дней истории: 1,913
Период для весов: 2016-03-28 — 2016-04-24


## 2. WRMSSE

Для каждого ряда считаем средний квадрат ошибки за 28 дней и делим на масштаб, рассчитанный по прошлым изменениям продаж. Начальные нули до первой продажи исключаем. Затем берём корень и усредняем с денежными весами по всем уровням.

Веса определяются выручкой последних 28 дней истории. На каждый из 12 уровней приходится одинаковая доля общего веса. Формула приведена в [руководстве M5](https://github.com/Mcompetitions/M5-methods).

В рабочем обсуждении независимый Python-пересчёт по формуле M5 отмечен как успешный: общий WRMSSE и все 12 уровней совпали. Служебную ячейку удалили после проверки; её вывод в этом экспорте не сохранён. Это сверка расчёта; серверная оценка Kaggle ещё не получена.


In [24]:
from scipy.sparse import csr_matrix


def calculate_rmsse_scale(history_values, aggregation, block_size=128):
    n_groups = aggregation.shape[0]
    n_days = history_values.shape[1]

    first_nonzero = np.full(n_groups, n_days, dtype=np.int32)
    squared_difference_sum = np.zeros(n_groups, dtype=np.float64)
    previous = np.zeros(n_groups, dtype=np.float64)

    for start in range(0, n_days, block_size):
        stop = min(start + block_size, n_days)

        block = np.asarray(
            aggregation @ history_values[:, start:stop],
            dtype=np.float64,
        )

        positive = block > 0
        newly_active = (first_nonzero == n_days) & positive.any(axis=1)

        if newly_active.any():
            first_nonzero[newly_active] = start + positive[newly_active].argmax(axis=1)

        differences = np.empty_like(block)
        differences[:, 0] = block[:, 0] - previous
        differences[:, 1:] = np.diff(block, axis=1)

        positions = np.arange(start, stop)
        exclude = positions[None, :] <= first_nonzero[:, None]
        differences[exclude] = 0.0

        squared_difference_sum += np.einsum(
            "ij,ij->i",
            differences,
            differences,
        )

        previous = block[:, -1].copy()

    difference_count = n_days - first_nonzero - 1

    return np.divide(
        squared_difference_sum,
        difference_count,
        out=np.zeros(n_groups, dtype=np.float64),
        where=difference_count > 0,
    )


class M5WRMSSE:
    def __init__(
        self,
        history,
        metadata,
        calendar,
        prices,
        levels,
        horizon=28,
    ):
        if len(metadata) != 30_490 or len(levels) != 12:
            raise ValueError("Для полного M5 нужны 30 490 рядов и 12 уровней.")

        self.series_index = pd.MultiIndex.from_frame(metadata[M5_KEYS])

        if (
            self.series_index.has_duplicates
            or history.index.has_duplicates
            or len(history) != len(self.series_index)
            or not self.series_index.isin(history.index).all()
        ):
            raise ValueError("Идентификаторы истории и metadata не совпадают.")

        history = history.reindex(self.series_index)

        if history.shape[1] < 28:
            raise ValueError("Для расчёта весов нужны минимум 28 дней.")

        if not isinstance(history.columns, pd.DatetimeIndex):
            raise TypeError("Столбцы истории должны содержать даты.")

        expected_history_dates = pd.date_range(
            history.columns[0],
            periods=history.shape[1],
            freq="D",
        )

        if not history.columns.equals(expected_history_dates):
            raise ValueError("В истории есть пропуски или нарушен порядок дней.")

        history_values = history.to_numpy(copy=False)

        if not np.isfinite(history_values).all() or (history_values < 0).any():
            raise ValueError(
                "История должна содержать конечные неотрицательные продажи."
            )

        self.horizon = horizon
        self.n_levels = len(levels)
        self.forecast_dates = pd.date_range(
            history.columns[-1] + pd.Timedelta(days=1),
            periods=horizon,
            freq="D",
            name="date",
        )

        recent_dates = history.columns[-28:]

        week_codes = (
            calendar.set_index("date").reindex(recent_dates)["wm_yr_wk"].to_numpy()
        )

        if pd.isna(week_codes).any():
            raise ValueError("В календаре отсутствуют даты для расчёта весов.")

        known_prices = prices["sell_price"].dropna().to_numpy()

        if not np.isfinite(known_prices).all() or (known_prices <= 0).any():
            raise ValueError("Известные цены должны быть конечными и положительными.")

        price_table = prices.pivot(
            index=M5_KEYS,
            columns="wm_yr_wk",
            values="sell_price",
        )

        daily_prices = price_table.reindex(
            index=self.series_index,
            columns=week_codes,
        ).to_numpy(dtype=np.float64)

        recent_units = history.iloc[:, -28:].to_numpy(dtype=np.float64)

        missing_price_with_sales = (recent_units > 0) & ~np.isfinite(daily_prices)

        if missing_price_with_sales.any():
            raise ValueError(
                "Есть положительные продажи без цены: веса нельзя посчитать корректно."
            )

        dollar_sales = (recent_units * np.nan_to_num(daily_prices, nan=0.0)).sum(axis=1)

        total_dollar_sales = dollar_sales.sum()

        if total_dollar_sales <= 0:
            raise ValueError("Суммарные денежные продажи равны нулю.")

        metadata = metadata.reset_index(drop=True)

        self.level_specs = {}
        level_rows = []

        for level_name, columns in levels.items():
            if columns:
                group_index = pd.MultiIndex.from_frame(metadata[columns])
                codes, groups = pd.factorize(
                    group_index,
                    sort=True,
                )
                n_groups = len(groups)
            else:
                codes = np.zeros(len(metadata), dtype=np.int32)
                n_groups = 1

            aggregation = csr_matrix(
                (
                    np.ones(len(metadata), dtype=np.float64),
                    (codes, np.arange(len(metadata))),
                ),
                shape=(n_groups, len(metadata)),
            )

            print(f"Масштабы: {level_name} — {n_groups:,} рядов")

            scale = calculate_rmsse_scale(
                history_values,
                aggregation,
            )

            group_dollar_sales = np.asarray(aggregation @ dollar_sales).ravel()

            weights = group_dollar_sales / total_dollar_sales / self.n_levels

            if ((scale <= 0) & (weights > 0)).any():
                raise ValueError(
                    f"{level_name}: есть ряд с положительным весом "
                    "и нулевым масштабом RMSSE."
                )

            self.level_specs[level_name] = {
                "aggregation": aggregation,
                "scale": scale,
                "weights": weights,
            }

            level_rows.append(
                {
                    "level": level_name,
                    "series": n_groups,
                    "weight_sum": weights.sum(),
                }
            )

        self.level_info = pd.DataFrame(level_rows).set_index("level")

        if self.level_info["series"].sum() != 42_840:
            raise ValueError("Получено неверное число рядов иерархии.")

        if not np.isclose(self.level_info["weight_sum"].sum(), 1.0):
            raise ValueError("Сумма весов WRMSSE отличается от единицы.")

    def _aligned_values(self, frame):
        if not isinstance(frame, pd.DataFrame):
            raise TypeError("Передай DataFrame с идентификаторами и датами.")

        if (
            frame.index.has_duplicates
            or len(frame) != len(self.series_index)
            or not self.series_index.isin(frame.index).all()
        ):
            raise ValueError("Набор товаров и магазинов не совпадает.")

        if not frame.columns.equals(self.forecast_dates):
            raise ValueError(
                "Столбцы должны содержать следующие 28 дат " "в правильном порядке."
            )

        values = frame.reindex(self.series_index).to_numpy(dtype=np.float64)

        if not np.isfinite(values).all():
            raise ValueError("В фактах или прогнозах есть NaN либо бесконечность.")

        return values

    def score(self, actual, prediction):
        actual_values = self._aligned_values(actual)
        prediction_values = self._aligned_values(prediction)

        bottom_errors = prediction_values - actual_values
        rows = []

        for level_name, spec in self.level_specs.items():
            aggregated_errors = spec["aggregation"] @ bottom_errors

            mse = np.mean(
                np.square(aggregated_errors),
                axis=1,
            )

            rmsse = np.sqrt(
                np.divide(
                    mse,
                    spec["scale"],
                    out=np.zeros_like(mse),
                    where=spec["scale"] > 0,
                )
            )

            contribution = float(np.dot(spec["weights"], rmsse))

            rows.append(
                {
                    "level": level_name,
                    "weighted_RMSSE": contribution * self.n_levels,
                    "contribution": contribution,
                }
            )

        report = pd.DataFrame(rows).set_index("level")
        overall = float(report["contribution"].sum())

        return overall, report


### 2.1. Веса и масштабы для public

Создаём оценщик по истории до 24 апреля. Проверяем число рядов и сумму весов. Для исторических окон те же величины рассчитываются по истории, доступной на их даты отсчёта.


In [25]:
wrmsse_evaluator = M5WRMSSE(
    history=train_daily,
    metadata=series_metadata,
    calendar=calendar,
    prices=prices_for_weights,
    levels=M5_LEVELS,
    horizon=HORIZON,
)

display(
    wrmsse_evaluator.level_info.style.format(
        {
            "series": "{:,.0f}",
            "weight_sum": "{:.6f}",
        }
    )
)

print(
    "Всего рядов:",
    f"{wrmsse_evaluator.level_info['series'].sum():,}",
)

print(
    "Сумма весов:",
    f"{wrmsse_evaluator.level_info['weight_sum'].sum():.6f}",
)

print(
    "Даты будущего прогноза:",
    wrmsse_evaluator.forecast_dates.min().date(),
    "—",
    wrmsse_evaluator.forecast_dates.max().date(),
)


Масштабы: Total — 1 рядов
Масштабы: State — 3 рядов
Масштабы: Store — 10 рядов
Масштабы: Category — 3 рядов
Масштабы: Department — 7 рядов
Масштабы: State / Category — 9 рядов
Масштабы: State / Department — 21 рядов
Масштабы: Store / Category — 30 рядов
Масштабы: Store / Department — 70 рядов
Масштабы: Item — 3,049 рядов
Масштабы: Item / State — 9,147 рядов
Масштабы: Item / Store — 30,490 рядов


,series,weight_sum
level,,
Total,1,0.083333
State,3,0.083333
Store,10,0.083333
Category,3,0.083333
Department,7,0.083333
State / Category,9,0.083333
State / Department,21,0.083333
Store / Category,30,0.083333
Store / Department,70,0.083333


Всего рядов: 42,840
Сумма весов: 1.000000
Даты будущего прогноза: 2016-04-25 — 2016-05-22


## 3. Признаки и обучающая выборка

Базовые признаки: средние за 7, 28 и 56 дней, разброс и доля нулей, изменение уровня продаж, среднее по дню недели и лаги 28–364 дня. Календарь и SNAP относятся к прогнозируемому дню. Продажи после даты отсчёта в признаки не попадают.


In [26]:
daily_categorical_features = [
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
]

daily_history = train_daily.reindex(wrmsse_evaluator.series_index)

daily_history_dates = daily_history.columns
daily_history_values = daily_history.to_numpy(copy=False)

daily_identifiers = (
    series_metadata.set_index(M5_KEYS)
    .loc[wrmsse_evaluator.series_index]
    .reset_index()
    .loc[:, daily_categorical_features]
)

daily_identifiers.index = wrmsse_evaluator.series_index

for column in daily_categorical_features:
    daily_identifiers[column] = daily_identifiers[column].astype("category")

daily_calendar = calendar.set_index("date")


def make_daily_features(forecast_origin, target_date):
    forecast_origin = pd.Timestamp(forecast_origin)
    target_date = pd.Timestamp(target_date)

    horizon = (target_date - forecast_origin).days

    if not 1 <= horizon <= HORIZON:
        raise ValueError("Горизонт должен быть от 1 до 28 дней.")

    origin_position = daily_history_dates.get_indexer([forecast_origin])[0]

    if origin_position < 55:
        raise ValueError(
            "Дата прогноза должна присутствовать в истории "
            "и иметь минимум 56 доступных дней."
        )

    X = daily_identifiers.copy()
    known_history = daily_history_values[:, : origin_position + 1]

    for window in (7, 28, 56):
        X[f"mean_{window}d"] = known_history[:, -window:].mean(axis=1).astype("float32")

    recent_28 = known_history[:, -28:]

    X["std_28d"] = recent_28.std(axis=1, ddof=1).astype("float32")

    X["zero_share_28d"] = (recent_28 == 0).mean(axis=1).astype("float32")

    X["trend_7d_28d"] = (X["mean_7d"] - X["mean_28d"]).astype("float32")

    recent_dates = daily_history_dates[origin_position - 27 : origin_position + 1]

    weekday_mask = recent_dates.dayofweek == target_date.dayofweek

    X["weekday_mean_4w"] = recent_28[:, weekday_mask].mean(axis=1).astype("float32")

    for lag in (28, 35, 42, 56, 364):
        source_date = target_date - pd.Timedelta(days=lag)

        if source_date > forecast_origin:
            raise ValueError("Лаг обращается к неизвестному будущему.")

        source_position = daily_history_dates.get_indexer([source_date])[0]

        if source_position >= 0:
            X[f"lag_{lag}d"] = daily_history_values[:, source_position].astype(
                "float32"
            )
        else:
            X[f"lag_{lag}d"] = np.float32(np.nan)

    calendar_values = daily_calendar.loc[target_date]

    target_features = {
        "horizon": horizon,
        "day_of_week": target_date.dayofweek,
        "day_of_month": target_date.day,
        "month": target_date.month,
        "week_of_year": int(target_date.isocalendar().week),
        "year": target_date.year,
    }

    for name, value in target_features.items():
        X[name] = np.int16(value)

    snap_by_state = {
        state: int(calendar_values[f"snap_{state}"]) for state in ("CA", "TX", "WI")
    }

    X["snap"] = X["state_id"].map(snap_by_state).astype("int8")

    X["has_event"] = np.int8(
        pd.notna(calendar_values["event_name_1"])
        or pd.notna(calendar_values["event_name_2"])
    )

    return X


### 3.1. План 52 × 2

Берём 52 еженедельные даты отсчёта. На каждой выбираем два горизонта из диапазона 1–28 и все товары с магазинами. Получается **3 170 960 строк**. Для последней даты оставляем горизонты 1 и 28.

Ниже определяются функции подготовки. Обучения в этом разделе нет.


In [27]:
def make_cb_training_plan(label_cutoff):
    label_cutoff = pd.Timestamp(label_cutoff)
    origins = pd.date_range(
        end=label_cutoff - pd.Timedelta(days=HORIZON), periods=CB_N_ORIGINS, freq="7D"
    )
    plan = []
    for i, origin in enumerate(origins):
        horizons = [5 * i % HORIZON + 1, (5 * i + HORIZON // 2) % HORIZON + 1]
        plan.append((origin, horizons))
    plan[-1] = (origins[-1], [1, HORIZON])
    return plan


### 3.2. История цены

Используем последнюю известную цену, её изменения за 1 и 4 недели, положение относительно истории за 4 и 13 недель и частоту наблюдения цены. Все эти признаки берутся на дату отсчёта. Отсутствие цены само по себе не доказывает отсутствие товара на складе.


In [28]:
price_data = pd.read_csv(
    DATA_RAW / "sell_prices.csv",
    dtype={
        "store_id": "category",
        "item_id": "category",
        "wm_yr_wk": "int32",
        "sell_price": "float64",
    },
)
price_week_starts = calendar.groupby("wm_yr_wk")["date"].min().sort_values()
price_week_order = price_week_starts.loc[price_week_starts <= cb_public_origin].index
price_data = price_data.loc[price_data["wm_yr_wk"].isin(price_week_order)].copy()
price_week_table = (
    price_data.pivot(index=M5_KEYS, columns="wm_yr_wk", values="sell_price")
    .reindex(index=wrmsse_evaluator.series_index, columns=price_week_order)
    .astype("float32")
)
price_last_known_table = price_week_table.ffill(axis=1)


@lru_cache(maxsize=64)
def price_features_at_origin(forecast_origin):
    forecast_origin = pd.Timestamp(forecast_origin)
    week = int(daily_calendar.loc[forecast_origin, "wm_yr_wk"])
    position = price_week_table.columns.get_loc(week)
    if position < 12:
        raise ValueError("Для признаков цены нужны 13 недель истории.")
    recent_4 = price_week_table.iloc[:, position - 3 : position + 1]
    recent_13 = price_week_table.iloc[:, position - 12 : position + 1]
    current = price_last_known_table.iloc[:, position]
    previous_1 = price_last_known_table.iloc[:, position - 1]
    previous_4 = price_last_known_table.iloc[:, position - 4]
    mean_4 = recent_4.mean(axis=1)
    mean_13 = recent_13.mean(axis=1)
    return pd.DataFrame(
        {
            "price_last_known": current,
            "price_observed_this_week": price_week_table.iloc[:, position].notna(),
            "price_change_1w": current / previous_1 - 1,
            "price_change_4w": current / previous_4 - 1,
            "price_vs_mean_4w": current / mean_4 - 1,
            "price_vs_mean_13w": current / mean_13 - 1,
            "price_vs_max_13w": current / recent_13.max(axis=1) - 1,
            "price_relative_std_13w": recent_13.std(axis=1, ddof=0) / mean_13,
            "price_observed_share_13w": recent_13.notna().mean(axis=1),
        }
    ).astype("float32")


def make_daily_features_with_price(forecast_origin, target_date):
    forecast_origin = pd.Timestamp(forecast_origin)
    base = make_daily_features(forecast_origin=forecast_origin, target_date=target_date)
    return base.join(price_features_at_origin(forecast_origin), validate="one_to_one")


#### Функция оценки

`price_evaluate` возвращает общий WRMSSE, ошибки по уровням и Bias. Bias считаем как отношение суммы прогнозов к сумме фактических продаж минус один. Отрицательное значение означает недопрогноз.


In [29]:
def price_evaluate(evaluator, actual, prediction):
    score, report = evaluator.score(actual, prediction)
    actual_total = (
        actual.reindex(index=prediction.index, columns=prediction.columns)
        .to_numpy(dtype=np.float64)
        .sum()
    )
    metrics = {
        "WRMSSE": float(score),
        "Bias": float(prediction.to_numpy().sum() / actual_total - 1),
    }
    return (metrics, report)


### 3.3. Регулярность продаж

Считаем время с последней продажи, долю дней с продажами, изменение этой доли и среднюю продажу среди ненулевых дней. Эти признаки помогают различать редкий спрос и изменение объёма обычных покупок.


In [30]:
@lru_cache(maxsize=64)
def regularity_features_at_origin(forecast_origin):
    forecast_origin = pd.Timestamp(forecast_origin)
    position = daily_history_dates.get_indexer([forecast_origin])[0]
    if position < 55:
        raise ValueError(
            "Дата должна присутствовать в истории и иметь минимум 56 доступных дней."
        )
    known = daily_history_values[:, : position + 1]
    positive_history = known > 0
    has_ever_sold = positive_history.any(axis=1)
    days_since_sale = positive_history[:, ::-1].argmax(axis=1).astype("float32")
    days_since_sale[~has_ever_sold] = np.nan
    recent_56 = known[:, -56:].astype("float64")
    recent_28 = recent_56[:, -28:]
    positive_56 = positive_history[:, -56:]
    positive_28 = positive_56[:, -28:]
    count_56 = positive_56.sum(axis=1)
    count_28 = positive_28.sum(axis=1)
    sum_56 = recent_56.sum(axis=1)
    sum_28 = recent_28.sum(axis=1)
    mean_positive_56 = np.divide(
        sum_56, count_56, out=np.full(len(known), np.nan), where=count_56 > 0
    )
    mean_positive_28 = np.divide(
        sum_28, count_28, out=np.full(len(known), np.nan), where=count_28 > 0
    )
    squared_sum_56 = np.einsum("ij,ij->i", recent_56, recent_56)
    correction = np.divide(
        np.square(sum_56), count_56, out=np.zeros(len(known)), where=count_56 > 0
    )
    positive_variance_56 = np.divide(
        np.maximum(squared_sum_56 - correction, 0.0),
        count_56 - 1,
        out=np.full(len(known), np.nan),
        where=count_56 > 1,
    )
    positive_cv2_56 = np.divide(
        positive_variance_56,
        np.square(mean_positive_56),
        out=np.full(len(known), np.nan),
        where=count_56 > 1,
    )
    frequency_7 = positive_56[:, -7:].mean(axis=1)
    frequency_previous_21 = positive_56[:, -28:-7].mean(axis=1)
    return pd.DataFrame(
        {
            "days_since_last_sale": days_since_sale,
            "has_never_sold": ~has_ever_sold,
            "sale_day_share_7d": frequency_7,
            "sale_day_share_56d": positive_56.mean(axis=1),
            "sale_frequency_change_7d_vs_prev21d": frequency_7 - frequency_previous_21,
            "mean_positive_sale_28d": mean_positive_28,
            "mean_positive_sale_56d": mean_positive_56,
            "positive_sale_cv2_56d": positive_cv2_56,
            "max_daily_sale_28d": recent_28.max(axis=1),
        },
        index=wrmsse_evaluator.series_index,
    ).astype("float32")


def make_daily_features_with_regularity(forecast_origin, target_date):
    forecast_origin = pd.Timestamp(forecast_origin)
    X = make_daily_features_with_price(forecast_origin, target_date)
    return X.join(regularity_features_at_origin(forecast_origin), validate="one_to_one")


### 3.4. Сборка выборки

Для каждой даты и горизонта строим признаки и берём целевую продажу нужного дня. Проверяем, что дата цели не выходит за конец обучающей истории. Таблицы объединяются в порядке «дата отсчёта → горизонт → товар и магазин».


In [31]:
def build_feature_dataset(label_cutoff, feature_function, feature_columns):
    label_cutoff = pd.Timestamp(label_cutoff)
    plan = make_cb_training_plan(label_cutoff)
    feature_parts = []
    target_parts = []
    for number, (origin, horizons) in enumerate(plan, start=1):
        for horizon in horizons:
            target_date = origin + pd.Timedelta(days=int(horizon))
            position = daily_history_dates.get_indexer([target_date])[0]
            if position < 0 or target_date > label_cutoff:
                raise ValueError(f"Недоступная целевая дата: {target_date}")
            X = feature_function(origin, target_date)
            feature_parts.append(X.loc[:, feature_columns].reset_index(drop=True))
            target_parts.append(daily_history_values[:, position].astype("float32"))
        if number % 13 == 0 or number == len(plan):
            print(f"Подготовлено дат: {number}/{len(plan)}")
    return (pd.concat(feature_parts, ignore_index=True), np.concatenate(target_parts))


### 3.5. Магазин и отдел

Добавляем общую динамику отдела в магазине: средние, тренд, разброс и недельный профиль. Также считаем долю товара в продажах отдела. Все статистики ограничены известной историей.


In [32]:
group_aggregation = wrmsse_evaluator.level_specs["Store / Department"]["aggregation"]
group_membership = group_aggregation.tocoo()
group_codes = np.empty(len(wrmsse_evaluator.series_index), dtype=np.int32)
group_codes[group_membership.col] = group_membership.row
group_daily_sales = np.asarray(
    group_aggregation @ daily_history_values, dtype=np.float64
)


def group_safe_ratio(numerator, denominator):
    numerator = np.asarray(numerator, dtype=np.float64)
    denominator = np.asarray(denominator, dtype=np.float64)
    return np.divide(
        numerator,
        denominator,
        out=np.full(numerator.shape, np.nan),
        where=denominator > 0,
    )


@lru_cache(maxsize=64)
def group_statistics_at_origin(forecast_origin):
    forecast_origin = pd.Timestamp(forecast_origin)
    position = daily_history_dates.get_indexer([forecast_origin])[0]
    if position < 55:
        raise ValueError("Для групповых признаков нужны 56 доступных дней.")
    known = group_daily_sales[:, : position + 1]
    recent_28 = known[:, -28:]
    recent_dates = daily_history_dates[position - 27 : position + 1]
    statistics = {
        f"mean_{window}d": known[:, -window:].mean(axis=1) for window in (7, 28, 56)
    }
    statistics["std_28d"] = recent_28.std(axis=1, ddof=1)
    statistics["weekday_mean_4w"] = np.column_stack(
        [
            recent_28[:, recent_dates.dayofweek == weekday].mean(axis=1)
            for weekday in range(7)
        ]
    )
    return statistics


def make_daily_features_with_groups(forecast_origin, target_date):
    forecast_origin = pd.Timestamp(forecast_origin)
    target_date = pd.Timestamp(target_date)
    X = make_daily_features_with_regularity(forecast_origin, target_date)
    statistics = group_statistics_at_origin(forecast_origin)
    mean_7 = statistics["mean_7d"][group_codes]
    mean_28 = statistics["mean_28d"][group_codes]
    mean_56 = statistics["mean_56d"][group_codes]
    weekday_mean = statistics["weekday_mean_4w"][group_codes, target_date.dayofweek]
    item_share_7 = group_safe_ratio(X["mean_7d"].to_numpy(), mean_7)
    item_share_28 = group_safe_ratio(X["mean_28d"].to_numpy(), mean_28)
    added_features = {
        "store_dept_mean_7d": mean_7,
        "store_dept_mean_28d": mean_28,
        "store_dept_mean_56d": mean_56,
        "store_dept_trend_7d_28d": group_safe_ratio(mean_7, mean_28) - 1,
        "store_dept_trend_28d_56d": group_safe_ratio(mean_28, mean_56) - 1,
        "store_dept_cv_28d": group_safe_ratio(
            statistics["std_28d"][group_codes], mean_28
        ),
        "store_dept_weekday_mean_4w": weekday_mean,
        "store_dept_weekday_factor_4w": group_safe_ratio(weekday_mean, mean_28),
        "item_share_store_dept_28d": item_share_28,
        "item_share_change_7d_28d": item_share_7 - item_share_28,
    }
    for column, values in added_features.items():
        X[column] = values.astype("float32")
    return X


### 3.6. Тот же товар в других магазинах

Считаем продажи товара в остальных магазинах, исключая текущий. Сравниваем уровни и тренды. После этого блока получаем итоговые 63 признака.


In [33]:
cross_store_aggregation = wrmsse_evaluator.level_specs["Item"]["aggregation"]
cross_store_membership = cross_store_aggregation.tocoo()
cross_store_codes = np.empty(len(wrmsse_evaluator.series_index), dtype=np.int32)
cross_store_codes[cross_store_membership.col] = cross_store_membership.row
cross_store_counts = np.asarray(cross_store_aggregation.sum(axis=1)).ravel()
cross_store_other_counts = cross_store_counts[cross_store_codes] - 1
cross_store_daily_totals = np.asarray(
    cross_store_aggregation @ daily_history_values, dtype=np.float64
)


@lru_cache(maxsize=64)
def cross_store_statistics_at_origin(forecast_origin):
    forecast_origin = pd.Timestamp(forecast_origin)
    position = daily_history_dates.get_indexer([forecast_origin])[0]
    if position < 55:
        raise ValueError("Для признаков других магазинов нужны 56 дней истории.")
    start = position - 55
    network_sales = cross_store_daily_totals[cross_store_codes, start : position + 1]
    own_sales = daily_history_values[:, start : position + 1]
    other_sales = (network_sales - own_sales) / cross_store_other_counts[:, None]
    recent_28 = other_sales[:, -28:]
    recent_dates = daily_history_dates[position - 27 : position + 1]
    statistics = {
        f"mean_{window}d": other_sales[:, -window:].mean(axis=1).astype("float32")
        for window in (7, 28, 56)
    }
    statistics["std_28d"] = recent_28.std(axis=1, ddof=1).astype("float32")
    statistics["weekday_mean_4w"] = np.column_stack(
        [
            recent_28[:, recent_dates.dayofweek == weekday].mean(axis=1)
            for weekday in range(7)
        ]
    ).astype("float32")
    return statistics


def make_daily_features_with_cross_store(forecast_origin, target_date):
    forecast_origin = pd.Timestamp(forecast_origin)
    target_date = pd.Timestamp(target_date)
    X = make_daily_features_with_groups(forecast_origin, target_date)
    statistics = cross_store_statistics_at_origin(forecast_origin)
    mean_7 = statistics["mean_7d"]
    mean_28 = statistics["mean_28d"]
    mean_56 = statistics["mean_56d"]
    weekday_mean = statistics["weekday_mean_4w"][:, target_date.dayofweek]
    own_trend = group_safe_ratio(X["mean_7d"].to_numpy(), X["mean_28d"].to_numpy()) - 1
    other_trend = group_safe_ratio(mean_7, mean_28) - 1
    added_features = {
        "other_stores_mean_7d": mean_7,
        "other_stores_mean_28d": mean_28,
        "other_stores_mean_56d": mean_56,
        "other_stores_trend_7d_28d": other_trend,
        "other_stores_trend_28d_56d": group_safe_ratio(mean_28, mean_56) - 1,
        "other_stores_cv_28d": group_safe_ratio(statistics["std_28d"], mean_28),
        "other_stores_weekday_mean_4w": weekday_mean,
        "other_stores_weekday_factor_4w": group_safe_ratio(weekday_mean, mean_28),
        "own_vs_other_stores_mean_28d": group_safe_ratio(
            X["mean_28d"].to_numpy(), mean_28
        ),
        "own_vs_other_stores_trend": own_trend - other_trend,
    }
    for column, values in added_features.items():
        X[column] = values.astype("float32")
    return X


## 4. Проверка подготовки

Строим признаки для одного будущего дня. Сверяем 63 столбца, их порядок и исходные файлы. Здесь также выводится сохранённый локальный ориентир: это результат предыдущего запуска на компьютере.


In [34]:
check_features = make_daily_features_with_cross_store(
    PUBLIC_ORIGIN, PUBLIC_ORIGIN + pd.Timedelta(days=1)
)
if check_features.columns.tolist() != FEATURE_COLUMNS:
    raise ValueError("Состав или порядок признаков изменился.")
if list(daily_categorical_features) != CATEGORY_COLUMNS:
    raise ValueError("Изменились категориальные признаки.")
if not check_features.index.equals(wrmsse_evaluator.series_index):
    raise ValueError("Изменился порядок товаров и магазинов.")
if daily_history_dates[-1] != PUBLIC_ORIGIN:
    raise ValueError("Обучающая история заканчивается другой датой.")

migration_check = {
    "status": "data_and_features_ready",
    "series": len(check_features),
    "features": check_features.shape[1],
    "training_rows": CB_N_ORIGINS * 2 * len(check_features),
    "runtime_versions": runtime_versions,
    "local_reference_public": LOCAL_REFERENCE["public_metrics"],
    "local_reference_cv": LOCAL_REFERENCE["mean_WRMSSE"],
    "public_training_enabled": RUN_PUBLIC,
    "cv_training_enabled": RUN_CV,
}
(RESULTS / "migration_check.json").write_text(
    json.dumps(migration_check, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Перенос данных и признаков проверен:", check_features.shape)
print("Обучающих строк на окно:", f"{migration_check['training_rows']:,}")
print("Локальный public WRMSSE:", LOCAL_REFERENCE["public_metrics"]["WRMSSE"])
print("Локальный средний CV WRMSSE:", LOCAL_REFERENCE["mean_WRMSSE"])
display(check_features.head())
del check_features
gc.collect()


Перенос данных и признаков проверен: (30490, 63)
Обучающих строк на окно: 3,170,960
Локальный public WRMSSE: 0.6367278319446078
Локальный средний CV WRMSSE: 0.6463928135345688


item_id    dept_id   cat_id store_id state_id  \
store_id item_id                                                              
CA_1     HOBBIES_1_001  HOBBIES_1_001  HOBBIES_1  HOBBIES     CA_1       CA   
         HOBBIES_1_002  HOBBIES_1_002  HOBBIES_1  HOBBIES     CA_1       CA   
         HOBBIES_1_003  HOBBIES_1_003  HOBBIES_1  HOBBIES     CA_1       CA   
         HOBBIES_1_004  HOBBIES_1_004  HOBBIES_1  HOBBIES     CA_1       CA   
         HOBBIES_1_005  HOBBIES_1_005  HOBBIES_1  HOBBIES     CA_1       CA   

                         mean_7d  mean_28d  mean_56d   std_28d  \
store_id item_id                                                 
CA_1     HOBBIES_1_001  1.142857  0.964286  0.982143  1.137969   
         HOBBIES_1_002  0.142857  0.071429  0.196429  0.262265   
         HOBBIES_1_003  0.857143  0.571429  0.553571  0.690066   
         HOBBIES_1_004  2.571429  1.821429  1.857143  1.886670   
         HOBBIES_1_005  1.714286  1.357143  1.232143  1.282771   

                        zero_share_28d  trend_7d_28d  weekday_mean_4w  \
store_id item_id                                                        
CA_1     HOBBIES_1_001        0.428571      0.178571             0.50   
         HOBBIES_1_002        0.928571      0.071429             0.25   
         HOBBIES_1_003        0.535714      0.285714             0.25   
         HOBBIES_1_004        0.321429      0.750000             1.75   
         HOBBIES_1_005        0.250000      0.357143             0.75   

                        lag_28d  lag_35d  lag_42d  ...  store_dept_cv_28d  \
store_id item_id                                   ...                      
CA_1     HOBBIES_1_001      1.0      1.0      0.0  ...           0.229824   
         HOBBIES_1_002      1.0      0.0      0.0  ...           0.229824   
         HOBBIES_1_003      0.0      0.0      0.0  ...           0.229824   
         HOBBIES_1_004      0.0      4.0      3.0  ...           0.229824   
         HOBBIES_1_005      1.0      2.0      2.0  ...           0.229824   

                        store_dept_weekday_mean_4w  \
store_id item_id                                     
CA_1     HOBBIES_1_001                      514.75   
         HOBBIES_1_002                      514.75   
         HOBBIES_1_003                      514.75   
         HOBBIES_1_004                      514.75   
         HOBBIES_1_005                      514.75   

                        store_dept_weekday_factor_4w  \
store_id item_id                                       
CA_1     HOBBIES_1_001                      1.030089   
         HOBBIES_1_002                      1.030089   
         HOBBIES_1_003                      1.030089   
         HOBBIES_1_004                      1.030089   
         HOBBIES_1_005                      1.030089   

                        item_share_store_dept_28d  item_share_change_7d_28d  \
store_id item_id                                                              
CA_1     HOBBIES_1_001                   0.001930                  0.000407   
         HOBBIES_1_002                   0.000143                  0.000149   
         HOBBIES_1_003                   0.001144                  0.000609   
         HOBBIES_1_004                   0.003645                  0.001612   
         HOBBIES_1_005                   0.002716                  0.000789   

                        other_stores_mean_7d  other_stores_mean_28d  \
store_id item_id                                                      
CA_1     HOBBIES_1_001              0.571429               0.468254   
         HOBBIES_1_002              0.333333               0.265873   
         HOBBIES_1_003              0.253968               0.265873   
         HOBBIES_1_004              1.603175               1.575397   
         HOBBIES_1_005              0.777778               0.876984   

                        other_stores_mean_56d  other_stores_trend_7d_28d  \
store_id item_id                                             

40

## 5. Перенос CatBoost

Используем параметры локального Optuna: глубина 4, скорость обучения 0.0336732, L2-регуляризация 1.5798164, 768 деревьев, Tweedie 1.5. Прогноз строится из одной даты сразу на все 28 дней. Факты внутри горизонта не подставляем.

Функции ниже собирают выборку, обучают модель, считают WRMSSE и Bias и сохраняют результаты в `/kaggle/working/demand-planner`. Факты public читаются из `d_1914`–`d_1941` только при оценке.


In [35]:
def clear_feature_caches():
    for function in (
        price_features_at_origin,
        regularity_features_at_origin,
        group_statistics_at_origin,
        cross_store_statistics_at_origin,
    ):
        if hasattr(function, "cache_clear"):
            function.cache_clear()
    gc.collect()


def actual_for_window(evaluator):
    dates = evaluator.forecast_dates
    if dates[-1] <= daily_history_dates[-1]:
        return daily_history.loc[evaluator.series_index, dates]
    if not dates.equals(wrmsse_evaluator.forecast_dates):
        raise ValueError("Поддерживается только известное public-окно.")
    day_columns = calendar.set_index("date").loc[dates, "d"].tolist()
    actual = pd.read_csv(
        DATA_RAW / "sales_train_evaluation.csv",
        usecols=[*M5_KEYS, *day_columns],
        dtype={
            **{key: "string" for key in M5_KEYS},
            **{column: "int32" for column in day_columns},
        },
    ).set_index(M5_KEYS)
    if (
        actual.index.has_duplicates
        or len(actual) != len(evaluator.series_index)
        or not evaluator.series_index.isin(actual.index).all()
    ):
        raise ValueError("Состав рядов в фактах не совпадает.")
    actual = actual.reindex(evaluator.series_index).loc[:, day_columns]
    actual.columns = dates
    return actual


def run_catboost_window(origin, label, reference_metrics):
    origin = pd.Timestamp(origin)
    started = perf_counter()
    output = RESULTS / label
    output.mkdir(parents=True, exist_ok=True)
    clear_feature_caches()

    X_train, y_train = build_feature_dataset(
        label_cutoff=origin,
        feature_function=make_daily_features_with_cross_store,
        feature_columns=FEATURE_COLUMNS,
    )
    train_rows = len(X_train)
    print(f"Обучающих строк: {train_rows:,}", flush=True)
    model = CatBoostRegressor(**CATBOOST_PARAMS, verbose=256)
    model.fit(
        X_train,
        y_train / TARGET_SCALE,
        cat_features=CATEGORY_COLUMNS,
        use_best_model=False,
    )
    del X_train, y_train
    clear_feature_caches()
    model_path = MODELS / f"{label}.cbm"
    model.save_model(str(model_path))

    if origin == PUBLIC_ORIGIN:
        evaluator = wrmsse_evaluator
    else:
        history = daily_history.loc[:, :origin]
        weeks = daily_calendar.loc[history.columns[-28:], "wm_yr_wk"].unique()
        with capture_output():
            evaluator = M5WRMSSE(
                history=history,
                metadata=series_metadata,
                calendar=calendar,
                prices=price_data.loc[price_data["wm_yr_wk"].isin(weeks)],
                levels=M5_LEVELS,
                horizon=HORIZON,
            )

    values = np.empty((len(evaluator.series_index), HORIZON), dtype=np.float64)
    for day_number, target_date in enumerate(evaluator.forecast_dates):
        X_day = make_daily_features_with_cross_store(origin, target_date).loc[
            :, FEATURE_COLUMNS
        ]
        if not X_day.index.equals(evaluator.series_index):
            raise ValueError("Нарушен порядок рядов прогноза.")
        values[:, day_number] = (
            model.predict(X_day, prediction_type="Exponent", thread_count=4)
            * TARGET_SCALE
        )
    prediction = pd.DataFrame(
        values, index=evaluator.series_index, columns=evaluator.forecast_dates
    )
    actual = actual_for_window(evaluator)
    metrics, report = price_evaluate(evaluator, actual, prediction)
    metrics.update(
        {
            "origin": str(origin.date()),
            "train_rows": train_rows,
            "seconds": perf_counter() - started,
        }
    )
    comparison = pd.DataFrame.from_dict(
        {
            "Локальный сохранённый контроль": reference_metrics,
            "Новый запуск Kaggle": {key: metrics[key] for key in ("WRMSSE", "Bias")},
        },
        orient="index",
    )
    comparison.index.name = "Источник"
    display(comparison.style.format({"WRMSSE": "{:.5f}", "Bias": "{:+.2%}"}))
    comparison.to_csv(output / "comparison.csv")
    report.to_csv(output / "metrics_by_level.csv")
    prediction.to_csv(output / "predictions.csv.gz", compression="gzip")
    config = {
        "params": CATBOOST_PARAMS,
        "features": FEATURE_COLUMNS,
        "categorical_features": CATEGORY_COLUMNS,
        "target_scale": TARGET_SCALE,
        "training_origins": CB_N_ORIGINS,
        "horizons_per_origin": 2,
        "runtime_versions": runtime_versions,
        "metrics": metrics,
        "model": str(model_path),
    }
    (output / "config.json").write_text(
        json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print("Результаты:", output, flush=True)
    del model, X_day, actual, prediction, values
    clear_feature_caches()
    return metrics


### 5.1. Public CatBoost: сохранённый запуск

При переносе получили **WRMSSE 0.63994**, Bias **−4.31%**. Локальный результат — **0.63673**, Bias **−4.20%**.

Исходные CSV совпали по SHA-256. В двух проверенных срезах различался один признак: `price_relative_std_13w`. Отключение pandas Bottleneck и очистка кеша цены устранили расхождение обоих срезов. Этот CatBoost обучен до исправления; его повторное обучение отложили.

Сейчас `RUN_PUBLIC = False`. Вывод ниже сохранён от завершённого запуска; при выполнении с текущим флагом обучение будет пропущено.


In [36]:
if RUN_PUBLIC:
    public_metrics = run_catboost_window(
        PUBLIC_ORIGIN,
        "public_catboost_optuna_v9_kaggle",
        LOCAL_REFERENCE["public_metrics"],
    )
else:
    print("Обучение public выключено. Для запуска установи RUN_PUBLIC = True.")


Подготовлено дат: 13/52
Подготовлено дат: 26/52
Подготовлено дат: 39/52
Подготовлено дат: 52/52
Обучающих строк: 3,170,960
0:	learn: 4.5620618	total: 1.7s	remaining: 21m 47s
256:	learn: 3.6685584	total: 6m 12s	remaining: 12m 20s
512:	learn: 3.6458410	total: 12m 38s	remaining: 6m 17s
767:	learn: 3.6361494	total: 19m 2s	remaining: 0us


,WRMSSE,Bias
Источник,,
Локальный сохранённый контроль,0.63673,-4.20%
Новый запуск Kaggle,0.63994,-4.31%


Результаты: /kaggle/working/demand-planner/results/m5_wrmsse/public_catboost_optuna_v9_kaggle


### 5.2. Три исторических окна CatBoost

Даты отсчёта — 31 января, 28 февраля и 27 марта 2016 года. В каждом случае прогнозируем следующие 28 дней. Средний WRMSSE Kaggle — **0.64957**, локальный — **0.64639**.

Это сохранённые запуски до исправления признака цены. Сейчас `RUN_CV = False`; для повторения трёх обучений флаг надо включить отдельно. Результаты public и исторических окон относятся к разным периодам.


In [37]:
if RUN_CV:
    cv_rows = []
    for fold_number, saved in enumerate(LOCAL_REFERENCE["folds"], start=1):
        print(f"Окно {fold_number}/3: {saved['origin']}", flush=True)
        metrics = run_catboost_window(
            saved["origin"],
            f"cv_{saved['origin']}_catboost_optuna_v9_kaggle",
            {key: saved[key] for key in ("WRMSSE", "Bias")},
        )
        cv_rows.append(metrics)
        pd.DataFrame(cv_rows).to_csv(RESULTS / "cv_metrics.csv", index=False)
    cv_table = pd.DataFrame(cv_rows).set_index("origin")
    display(
        cv_table.style.format(
            {"WRMSSE": "{:.5f}", "Bias": "{:+.2%}", "seconds": "{:.0f}"}
        )
    )
    print("Средний WRMSSE Kaggle:", f"{cv_table['WRMSSE'].mean():.5f}")
else:
    print("Обучение исторических окон выключено: RUN_CV = False.")


Окно 1/3: 2016-01-31
Подготовлено дат: 13/52
Подготовлено дат: 26/52
Подготовлено дат: 39/52
Подготовлено дат: 52/52
Обучающих строк: 3,170,960
0:	learn: 4.4902125	total: 1.61s	remaining: 20m 34s
256:	learn: 3.6090978	total: 6m 16s	remaining: 12m 29s
512:	learn: 3.5854853	total: 12m 47s	remaining: 6m 21s
767:	learn: 3.5742800	total: 19m 16s	remaining: 0us


,WRMSSE,Bias
Источник,,
Локальный сохранённый контроль,0.72067,-3.19%
Новый запуск Kaggle,0.72550,-3.91%


Результаты: /kaggle/working/demand-planner/results/m5_wrmsse/cv_2016-01-31_catboost_optuna_v9_kaggle
Окно 2/3: 2016-02-28
Подготовлено дат: 13/52
Подготовлено дат: 26/52
Подготовлено дат: 39/52
Подготовлено дат: 52/52
Обучающих строк: 3,170,960
0:	learn: 4.5063630	total: 1.62s	remaining: 20m 40s
256:	learn: 3.6246442	total: 6m 14s	remaining: 12m 24s
512:	learn: 3.6027819	total: 12m 45s	remaining: 6m 20s
767:	learn: 3.5911515	total: 19m 12s	remaining: 0us


,WRMSSE,Bias
Источник,,
Локальный сохранённый контроль,0.65168,-0.27%
Новый запуск Kaggle,0.65715,-0.56%


Результаты: /kaggle/working/demand-planner/results/m5_wrmsse/cv_2016-02-28_catboost_optuna_v9_kaggle
Окно 3/3: 2016-03-27
Подготовлено дат: 13/52
Подготовлено дат: 26/52
Подготовлено дат: 39/52
Подготовлено дат: 52/52
Обучающих строк: 3,170,960
0:	learn: 4.5143381	total: 1.61s	remaining: 20m 37s
256:	learn: 3.6340243	total: 6m 17s	remaining: 12m 31s
512:	learn: 3.6104412	total: 12m 48s	remaining: 6m 21s
767:	learn: 3.5997249	total: 19m 15s	remaining: 0us


,WRMSSE,Bias
Источник,,
Локальный сохранённый контроль,0.56683,-1.66%
Новый запуск Kaggle,0.56605,-1.77%


Результаты: /kaggle/working/demand-planner/results/m5_wrmsse/cv_2016-03-27_catboost_optuna_v9_kaggle


,WRMSSE,Bias,train_rows,seconds
origin,,,,
2016-01-31,0.72550,-3.91%,3170960,1189
2016-02-28,0.65715,-0.56%,3170960,1185
2016-03-27,0.56605,-1.77%,3170960,1188


Средний WRMSSE Kaggle: 0.64957


## 6. LightGBM на прежней выборке 52 × 2

Оставляем те же 63 признака и обучающие даты. Для категориальных столбцов фиксируем одинаковые справочники при обучении и прогнозе. Используем Tweedie 1.5, learning rate 0.05, 63 листа и 4 CPU-потока.

С этого блока Bottleneck отключён через `pd.set_option("compute.use_bottleneck", False)`, кеш цены очищен. Проверка отсутствия файла public выполняется до долгого обучения.


In [42]:
import gc
import io
import json
from contextlib import redirect_stdout
from pathlib import Path
from time import perf_counter

import lightgbm as lgb
import numpy as np
import pandas as pd
from IPython.display import display

lg_required = [
    "DATA_RAW",
    "HORIZON",
    "M5_KEYS",
    "M5_LEVELS",
    "M5WRMSSE",
    "daily_history",
    "daily_identifiers",
    "daily_calendar",
    "calendar",
    "series_metadata",
    "price_data",
    "price_evaluate",
    "price_features_at_origin",
    "make_cb_training_plan",
    "make_daily_features_with_cross_store",
    "build_feature_dataset",
]

lg_missing = [name for name in lg_required if name not in globals()]

if lg_missing:
    raise RuntimeError("Сначала нужны ячейки подготовки: " + ", ".join(lg_missing))

lg_public_labels_path = Path(DATA_RAW) / "sales_train_evaluation.csv"

if not lg_public_labels_path.is_file():
    raise FileNotFoundError(lg_public_labels_path)

if HORIZON != 28:
    raise ValueError("Для этого опыта нужен горизонт 28 дней.")

pd.set_option("compute.use_bottleneck", False)
price_features_at_origin.cache_clear()

RUN_PUBLIC = False
RUN_CV = False

lg_root = Path("/kaggle/working/demand-planner")
lg_results = lg_root / "results/m5_wrmsse/lightgbm_v1_bn_off"
lg_models = lg_root / "models/lightgbm_v1_bn_off"

lg_results.mkdir(parents=True, exist_ok=True)
lg_models.mkdir(parents=True, exist_ok=True)

lg_public_origin = pd.Timestamp("2016-04-24")
lg_historical_origin = pd.Timestamp("2016-03-27")

if daily_history.columns[-1] != lg_public_origin:
    raise ValueError("История должна заканчиваться 2016-04-24.")

lg_feature_columns = make_daily_features_with_cross_store(
    lg_historical_origin,
    lg_historical_origin + pd.Timedelta(days=1),
).columns.tolist()

lg_categorical_columns = ["item_id", "dept_id", "cat_id", "store_id", "state_id"]

if len(lg_feature_columns) != 63:
    raise ValueError("Ожидались прежние 63 признака.")

lg_category_types = {
    column: pd.CategoricalDtype(
        categories=daily_identifiers[column].cat.categories,
        ordered=False,
    )
    for column in lg_categorical_columns
}

lg_params = {
    "objective": "tweedie",
    "tweedie_variance_power": 1.5,
    "metric": "tweedie",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_child_samples": 100,
    "reg_lambda": 1.0,
    "colsample_bytree": 0.9,
    "subsample": 0.8,
    "subsample_freq": 1,
    "max_bin": 255,
    "random_state": 42,
    "n_jobs": 4,
    "device_type": "cpu",
    "deterministic": True,
    "force_col_wise": True,
    "verbosity": -1,
}


def lg_prepare_categories(frame):
    for column, dtype in lg_category_types.items():
        frame[column] = frame[column].astype(dtype)

        if frame[column].isna().any():
            raise ValueError(f"Неизвестная категория в {column}.")

    return frame


def lg_training_data(origin):
    plan = make_cb_training_plan(origin)

    if len(plan) != 52 or any(len(horizons) != 2 for _, horizons in plan):
        raise ValueError("Нужна прежняя схема 52 × 2.")

    X, y = build_feature_dataset(
        label_cutoff=origin,
        feature_function=make_daily_features_with_cross_store,
        feature_columns=lg_feature_columns,
    )

    if len(X) != 3_170_960:
        raise ValueError("Изменилось количество обучающих примеров.")

    return lg_prepare_categories(X), y


def lg_evaluator(origin):
    history = daily_history.loc[:, :origin]

    weeks = daily_calendar.loc[history.columns[-28:], "wm_yr_wk"].unique()

    with redirect_stdout(io.StringIO()):
        evaluator = M5WRMSSE(
            history=history,
            metadata=series_metadata,
            calendar=calendar,
            prices=price_data.loc[price_data["wm_yr_wk"].isin(weeks)],
            levels=M5_LEVELS,
            horizon=HORIZON,
        )

    return evaluator


def lg_forecast_data(origin, evaluator):
    parts = []

    for date in evaluator.forecast_dates:
        part = make_daily_features_with_cross_store(origin, date)

        if not part.index.equals(evaluator.series_index):
            raise ValueError("Нарушен порядок товаров и магазинов.")

        parts.append(part.loc[:, lg_feature_columns].reset_index(drop=True))

    return lg_prepare_categories(pd.concat(parts, ignore_index=True))


def lg_prediction_frame(model, X, evaluator):
    values = model.predict(X)

    if not np.isfinite(values).all() or (values < 0).any():
        raise ValueError("Некорректные значения прогноза.")

    return pd.DataFrame(
        values.reshape(HORIZON, len(evaluator.series_index)).T,
        index=evaluator.series_index,
        columns=evaluator.forecast_dates,
    )


def lg_save_result(name, origin, model, actual, prediction, evaluator, seconds):
    metrics, levels = price_evaluate(evaluator, actual, prediction)

    directory = lg_results / name
    directory.mkdir(parents=True, exist_ok=True)

    tree_count = int(model.best_iteration_ or model.n_estimators_)

    model.booster_.save_model(
        str(lg_models / f"{name}.txt"),
        num_iteration=tree_count,
    )

    prediction.to_csv(
        directory / "predictions.csv.gz",
        compression="gzip",
    )
    levels.to_csv(directory / "metrics_by_level.csv")

    config = {
        "origin": str(origin.date()),
        "params": {**lg_params, "n_estimators": tree_count},
        "features": lg_feature_columns,
        "categorical_features": lg_categorical_columns,
        "categories": {
            column: [str(value) for value in dtype.categories]
            for column, dtype in lg_category_types.items()
        },
        "training_origins": 52,
        "horizons_per_origin": 2,
        "train_rows": 3_170_960,
        "compute_use_bottleneck": False,
        "lightgbm_version": lgb.__version__,
        "pandas_version": pd.__version__,
        "numpy_version": np.__version__,
        "seconds": seconds,
        "metrics": metrics,
    }

    (directory / "config.json").write_text(
        json.dumps(config, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    pd.DataFrame([metrics]).to_csv(
        directory / "metrics.csv",
        index=False,
    )

    return metrics


print("LightGBM:", lgb.__version__)
print("Признаков:", len(lg_feature_columns))
print("Bottleneck:", pd.get_option("compute.use_bottleneck"))
print("Будет два обучения: историческое окно, затем public.")


LightGBM: 4.6.0
Признаков: 63
Bottleneck: False
Будет два обучения: историческое окно, затем public.


### 6.1. Первый выбор числа деревьев: Tweedie

Обучаемся по истории до 27 марта и используем следующие 28 дней для ранней остановки. По Tweedie выбрано **52 дерева**. Получили WRMSSE **0.95840**, Bias **−9.99%**.

Этот опыт сохраняем как часть истории: его оценщик, факты и прогноз используются в проверке следующего блока. Удалять ячейку отдельно от этих зависимостей пока нельзя.


In [43]:
lg_started = perf_counter()

lg_hist_evaluator = lg_evaluator(lg_historical_origin)
lg_X_train, lg_y_train = lg_training_data(lg_historical_origin)
lg_X_valid = lg_forecast_data(lg_historical_origin, lg_hist_evaluator)

lg_hist_actual = daily_history.reindex(
    index=lg_hist_evaluator.series_index,
    columns=lg_hist_evaluator.forecast_dates,
)

lg_y_valid = lg_hist_actual.to_numpy(dtype=np.float32).T.reshape(-1)

print(f"Обучающих строк: {len(lg_X_train):,}")
print(f"Проверочных строк: {len(lg_X_valid):,}")

lg_hist_model = lgb.LGBMRegressor(
    **lg_params,
    n_estimators=3000,
)

lg_hist_model.fit(
    lg_X_train,
    lg_y_train,
    categorical_feature=lg_categorical_columns,
    eval_set=[(lg_X_valid, lg_y_valid)],
    callbacks=[
        lgb.early_stopping(100, first_metric_only=True),
        lgb.log_evaluation(200),
    ],
)

lg_tree_count = int(lg_hist_model.best_iteration_ or lg_hist_model.n_estimators_)

lg_hist_prediction = lg_prediction_frame(
    lg_hist_model,
    lg_X_valid,
    lg_hist_evaluator,
)

lg_hist_metrics = lg_save_result(
    "historical_2016-03-27",
    lg_historical_origin,
    lg_hist_model,
    lg_hist_actual,
    lg_hist_prediction,
    lg_hist_evaluator,
    perf_counter() - lg_started,
)

print("Выбрано деревьев:", lg_tree_count)

if lg_tree_count >= 3000:
    print("Выбран верхний лимит — позже проверим больше итераций.")

print("Это окно использовано для early stopping.")
print("Сравнение со старым CatBoost предварительное.")

display(
    pd.DataFrame.from_dict(
        {
            "CatBoost — прежний Kaggle": {
                "WRMSSE": 0.56605,
                "Bias": -0.0177,
            },
            "LightGBM — Bottleneck выключен": lg_hist_metrics,
        },
        orient="index",
    ).style.format(
        {
            "WRMSSE": "{:.5f}",
            "Bias": "{:+.2%}",
        }
    )
)

del lg_X_train, lg_y_train, lg_X_valid, lg_y_valid
gc.collect()


Подготовлено дат: 13/52
Подготовлено дат: 26/52
Подготовлено дат: 39/52
Подготовлено дат: 52/52
Обучающих строк: 3,170,960
Проверочных строк: 853,720
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[52]	valid_0's tweedie: 3.87854
Evaluated only: tweedie
Выбрано деревьев: 52
Это окно использовано для early stopping.
Сравнение со старым CatBoost предварительное.


,WRMSSE,Bias
CatBoost — прежний Kaggle,0.56605,-1.77%
LightGBM — Bottleneck выключен,0.95840,-9.99%


122

### 6.2. Выбор числа деревьев по WRMSSE

Меняем критерий ранней остановки на WRMSSE, сохраняя остальные настройки и выборку. Функция потерь обучения остаётся Tweedie. После 100 итераций без улучшения выбираются **386 деревьев**: WRMSSE **0.55498**, Bias **−2.20%**.

Первое сообщение `Проверка расчёта WRMSSE: 0.95840` сверяет callback с нашим оценщиком на прежнем прогнозе. Независимая проверка формулы выполнена позже отдельной служебной ячейкой.

Это окно участвовало в выборе числа деревьев. Сравнение с CatBoost здесь предварительное; следующая проверка — перенос выбранных 386 деревьев на public.


In [46]:
import gc
import json
from time import perf_counter

lg_v2_actual = lg_hist_actual.reindex(
    index=lg_hist_evaluator.series_index,
    columns=lg_hist_evaluator.forecast_dates,
).to_numpy(dtype=np.float64)


def lg_wrmsse_metric(y_true, y_pred):
    errors = y_pred.reshape(HORIZON, lg_v2_actual.shape[0]).T - lg_v2_actual
    score = 0.0

    for spec in lg_hist_evaluator.level_specs.values():
        aggregated = spec["aggregation"] @ errors
        mse = np.mean(np.square(aggregated), axis=1)

        rmsse = np.sqrt(
            np.divide(
                mse,
                spec["scale"],
                out=np.zeros_like(mse),
                where=spec["scale"] > 0,
            )
        )

        score += float(np.dot(spec["weights"], rmsse))

    return "WRMSSE", score, False


lg_v2_y_valid = lg_v2_actual.T.reshape(-1)

lg_v2_old_values = (
    lg_hist_prediction.reindex(
        index=lg_hist_evaluator.series_index,
        columns=lg_hist_evaluator.forecast_dates,
    )
    .to_numpy(dtype=np.float64)
    .T.reshape(-1)
)

lg_v2_check = lg_wrmsse_metric(
    lg_v2_y_valid,
    lg_v2_old_values,
)[1]

if not np.isclose(
    lg_v2_check,
    lg_hist_metrics["WRMSSE"],
    rtol=1e-9,
):
    raise ValueError("Расчёт WRMSSE не совпал с прежним оценщиком.")

print(f"Проверка расчёта WRMSSE: {lg_v2_check:.5f}")

pd.set_option("compute.use_bottleneck", False)
price_features_at_origin.cache_clear()

lg_params = {**lg_params, "metric": "None"}

lg_results = lg_root / "results/m5_wrmsse/lightgbm_v2_wrmsse"
lg_models = lg_root / "models/lightgbm_v2_wrmsse"

lg_results.mkdir(parents=True, exist_ok=True)
lg_models.mkdir(parents=True, exist_ok=True)

lg_v2_started = perf_counter()

lg_v2_X_train, lg_v2_y_train = lg_training_data(lg_historical_origin)
lg_v2_X_valid = lg_forecast_data(
    lg_historical_origin,
    lg_hist_evaluator,
)

lg_hist_model_v2 = lgb.LGBMRegressor(
    **lg_params,
    n_estimators=3000,
)

lg_hist_model_v2.fit(
    lg_v2_X_train,
    lg_v2_y_train,
    categorical_feature=lg_categorical_columns,
    eval_set=[(lg_v2_X_valid, lg_v2_y_valid)],
    eval_metric=lg_wrmsse_metric,
    callbacks=[
        lgb.early_stopping(100, first_metric_only=True),
        lgb.log_evaluation(100),
    ],
)

lg_tree_count = int(lg_hist_model_v2.best_iteration_ or lg_hist_model_v2.n_estimators_)

lg_hist_prediction_v2 = lg_prediction_frame(
    lg_hist_model_v2,
    lg_v2_X_valid,
    lg_hist_evaluator,
)

lg_hist_metrics_v2 = lg_save_result(
    "historical_2016-03-27",
    lg_historical_origin,
    lg_hist_model_v2,
    lg_hist_actual,
    lg_hist_prediction_v2,
    lg_hist_evaluator,
    perf_counter() - lg_v2_started,
)

lg_v2_config_path = lg_results / "historical_2016-03-27/config.json"
lg_v2_config = json.loads(lg_v2_config_path.read_text())

lg_v2_config.update(
    {
        "early_stopping_metric": "WRMSSE",
        "early_stopping_rounds": 100,
        "max_iterations": 3000,
        "selection_window_used_for_early_stopping": True,
    }
)

lg_v2_config_path.write_text(
    json.dumps(lg_v2_config, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Выбрано деревьев по WRMSSE:", lg_tree_count)

if lg_tree_count >= 3000:
    print("Выбран верхний лимит; больше деревьев пока не проверено.")

lg_v2_comparison = pd.DataFrame.from_dict(
    {
        "CatBoost — прежний Kaggle": {
            "WRMSSE": 0.56605,
            "Bias": -0.0177,
        },
        "LightGBM — выбор по Tweedie": lg_hist_metrics,
        "LightGBM — выбор по WRMSSE": lg_hist_metrics_v2,
    },
    orient="index",
)

lg_v2_comparison.to_csv(lg_results / "historical_comparison.csv")

display(
    lg_v2_comparison.style.format(
        {
            "WRMSSE": "{:.5f}",
            "Bias": "{:+.2%}",
        }
    )
)

del lg_v2_X_train, lg_v2_y_train, lg_v2_X_valid
gc.collect()


Проверка расчёта WRMSSE: 0.95840
Подготовлено дат: 13/52
Подготовлено дат: 26/52
Подготовлено дат: 39/52
Подготовлено дат: 52/52
Training until validation scores don't improve for 100 rounds
[100]	valid_0's WRMSSE: 0.646916
[200]	valid_0's WRMSSE: 0.562359
[300]	valid_0's WRMSSE: 0.55734
[400]	valid_0's WRMSSE: 0.555903
Early stopping, best iteration is:
[386]	valid_0's WRMSSE: 0.554977
Evaluated only: WRMSSE
Выбрано деревьев по WRMSSE: 386


,WRMSSE,Bias
CatBoost — прежний Kaggle,0.56605,-1.77%
LightGBM — выбор по Tweedie,0.95840,-9.99%
LightGBM — выбор по WRMSSE,0.55498,-2.20%


32

### 6.3. Public с фиксированными 386 деревьями

Обучаем новую модель по истории до 24 апреля. Число деревьев уже выбрано, public-факты в fit не передаются. После прогноза читаем фактические продажи и считаем метрики.

Получили **WRMSSE 0.67879**, Bias **−5.98%** против **0.63994** и **−4.31%** у прежнего Kaggle CatBoost. На этом периоде LightGBM уступил. CatBoost в таблице обучен до исправления цены, LightGBM — после.


In [47]:
lg_started = perf_counter()

lg_public_evaluator = lg_evaluator(lg_public_origin)
lg_X_train, lg_y_train = lg_training_data(lg_public_origin)

lg_public_model = lgb.LGBMRegressor(
    **lg_params,
    n_estimators=lg_tree_count,
)

lg_public_model.fit(
    lg_X_train,
    lg_y_train,
    categorical_feature=lg_categorical_columns,
    callbacks=[lgb.log_evaluation(200)],
)

del lg_X_train, lg_y_train
gc.collect()

lg_X_public = lg_forecast_data(
    lg_public_origin,
    lg_public_evaluator,
)

lg_public_prediction = lg_prediction_frame(
    lg_public_model,
    lg_X_public,
    lg_public_evaluator,
)

del lg_X_public
gc.collect()

lg_day_columns = (
    calendar.set_index("date").loc[lg_public_evaluator.forecast_dates, "d"].tolist()
)

lg_public_actual = pd.read_csv(
    lg_public_labels_path,
    usecols=[*M5_KEYS, *lg_day_columns],
    dtype={
        **{column: "string" for column in M5_KEYS},
        **{column: "int32" for column in lg_day_columns},
    },
).set_index(M5_KEYS)

if (
    lg_public_actual.index.has_duplicates
    or len(lg_public_actual) != len(lg_public_evaluator.series_index)
    or not lg_public_evaluator.series_index.isin(lg_public_actual.index).all()
):
    raise ValueError("Состав рядов в фактах public не совпадает.")

lg_public_actual = lg_public_actual.reindex(lg_public_evaluator.series_index).loc[
    :, lg_day_columns
]

lg_public_actual.columns = lg_public_evaluator.forecast_dates

lg_public_metrics = lg_save_result(
    "public",
    lg_public_origin,
    lg_public_model,
    lg_public_actual,
    lg_public_prediction,
    lg_public_evaluator,
    perf_counter() - lg_started,
)

lg_comparison = pd.DataFrame.from_dict(
    {
        "CatBoost — прежний Kaggle": {
            "WRMSSE": 0.63994,
            "Bias": -0.0431,
        },
        "LightGBM — Bottleneck выключен": lg_public_metrics,
    },
    orient="index",
)

lg_comparison.to_csv(lg_results / "public_comparison.csv")

display(
    lg_comparison.style.format(
        {
            "WRMSSE": "{:.5f}",
            "Bias": "{:+.2%}",
        }
    )
)

print("Модели:", lg_models)
print("Результаты:", lg_results)


Подготовлено дат: 13/52
Подготовлено дат: 26/52
Подготовлено дат: 39/52
Подготовлено дат: 52/52


,WRMSSE,Bias
CatBoost — прежний Kaggle,0.63994,-4.31%
LightGBM — Bottleneck выключен,0.67879,-5.98%


Модели: /kaggle/working/demand-planner/models/lightgbm_v2_wrmsse
Результаты: /kaggle/working/demand-planner/results/m5_wrmsse/lightgbm_v2_wrmsse


## 7. Плотность: 52 × 2 → 52 × 4

Сохраняем все прежние примеры и добавляем два горизонта на каждую дату отсчёта. Получается **6 341 920 строк**. Выборку один раз готовим для обеих моделей.

Признаки и параметры фиксированы: CatBoost — 768 деревьев, LightGBM — 386. Ранней остановки в этом опыте нет. Проверяем одно историческое окно, **28 марта — 24 апреля**; public для плотных моделей пока не запускается.

CatBoost сравниваем с локальным контролем 52 × 2, как было решено перед опытом. LightGBM сравниваем с его Kaggle-контролем. После каждой модели сохраняются файлы и обновляется общий ZIP моделей и результатов.


In [49]:
import gc
import json
from os.path import relpath
from pathlib import Path
from time import perf_counter
from zipfile import ZIP_DEFLATED, ZipFile

import catboost
import lightgbm as lgb
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from IPython.display import FileLink, display

dense_required = [
    "make_cb_training_plan",
    "make_daily_features_with_cross_store",
    "daily_history",
    "daily_history_dates",
    "daily_history_values",
    "price_features_at_origin",
    "price_evaluate",
    "lg_feature_columns",
    "lg_categorical_columns",
    "lg_prepare_categories",
    "lg_forecast_data",
    "lg_hist_evaluator",
    "lg_hist_model_v2",
    "lg_hist_metrics_v2",
]

dense_missing = [name for name in dense_required if name not in globals()]
if dense_missing:
    raise RuntimeError("Нужны переменные прежних блоков: " + ", ".join(dense_missing))

pd.set_option("compute.use_bottleneck", False)
price_features_at_origin.cache_clear()

dense_origin = pd.Timestamp("2016-03-27")
dense_evaluator = lg_hist_evaluator
dense_columns = list(lg_feature_columns)
dense_categories = list(lg_categorical_columns)
dense_lg_trees = int(lg_hist_model_v2.best_iteration_ or lg_hist_model_v2.n_estimators_)

if len(dense_columns) != 63 or dense_lg_trees != 386:
    raise ValueError("Ожидались прежние 63 признака и 386 деревьев LightGBM.")

if not dense_evaluator.forecast_dates.equals(
    pd.date_range("2016-03-28", periods=28, name="date")
):
    raise ValueError("Нужно историческое окно 2016-03-28 — 2016-04-24.")

if not daily_history.index.equals(dense_evaluator.series_index):
    raise ValueError("Изменился порядок рядов в истории.")

dense_cb_params = {
    "loss_function": "Tweedie:variance_power=1.5",
    "eval_metric": "Tweedie:variance_power=1.5",
    "random_seed": 42,
    "thread_count": 4,
    "allow_writing_files": False,
    "depth": 4,
    "learning_rate": 0.03367321864197597,
    "l2_leaf_reg": 1.579816405580968,
    "iterations": 768,
}

dense_lg_params = {
    **lg_hist_model_v2.get_params(),
    "n_estimators": 386,
}

dense_controls = {
    "CatBoost": {
        "WRMSSE": 0.5668285787684458,
        "Bias": -0.016609536359959987,
    },
    "LightGBM": {key: float(lg_hist_metrics_v2[key]) for key in ("WRMSSE", "Bias")},
}

dense_root = Path("/kaggle/working/demand-planner")
dense_results = dense_root / "results/m5_wrmsse/density_52x4_v1/2016-03-27"
dense_models = dense_root / "models/density_52x4_v1/2016-03-27"
dense_results.mkdir(parents=True, exist_ok=True)
dense_models.mkdir(parents=True, exist_ok=True)

dense_base_plan = make_cb_training_plan(dense_origin)
dense_expected_origins = pd.date_range(
    end=dense_origin - pd.Timedelta(days=28),
    periods=52,
    freq="7D",
)

if len(dense_base_plan) != 52 or any(
    pd.Timestamp(origin) != expected or len(set(horizons)) != 2
    for (origin, horizons), expected in zip(dense_base_plan, dense_expected_origins)
):
    raise ValueError("Изменился прежний план 52 × 2.")

if set(dense_base_plan[-1][1]) != {1, 28}:
    raise ValueError("У последней даты прежнего плана ожидались горизонты 1 и 28.")

dense_plan = []
dense_plan_rows = []

for number, (origin, old_horizons) in enumerate(dense_base_plan):
    first = int(old_horizons[0])
    extra = [(first - 1 + shift) % 28 + 1 for shift in (7, 21)]

    if number == 51:
        extra = [10, 19]

    horizons = sorted(set(map(int, old_horizons)) | set(extra))

    if len(horizons) != 4 or not all(1 <= h <= 28 for h in horizons):
        raise ValueError("Не удалось построить четыре разных горизонта.")

    dense_plan.append((pd.Timestamp(origin), horizons))

    for horizon in horizons:
        dense_plan_rows.append(
            {
                "origin": str(pd.Timestamp(origin).date()),
                "horizon": horizon,
                "target_date": str(
                    (pd.Timestamp(origin) + pd.Timedelta(days=horizon)).date()
                ),
                "present_in_52x2": horizon in old_horizons,
            }
        )

pd.DataFrame(dense_plan_rows).to_csv(
    dense_results / "training_plan.csv",
    index=False,
)

dense_protocol = {
    "origin": str(dense_origin.date()),
    "training_origins": 52,
    "horizons_per_origin": 4,
    "train_rows": 6_341_920,
    "features": dense_columns,
    "categorical_features": dense_categories,
    "target_scale": 1.0,
    "compute_use_bottleneck": False,
    "catboost_params": dense_cb_params,
    "lightgbm_params": dense_lg_params,
    "controls_52x2": dense_controls,
    "catboost_control_source": ("saved local Optuna fold; no matched Kaggle rerun"),
    "lightgbm_control_source": ("Kaggle historical run selected by WRMSSE"),
    "versions": {
        "catboost": catboost.__version__,
        "lightgbm": lgb.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
    },
}

(dense_results / "protocol.json").write_text(
    json.dumps(dense_protocol, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

dense_comparison_path = dense_results / "comparison.csv"
dense_rows = (
    pd.read_csv(dense_comparison_path, index_col=0).to_dict("index")
    if dense_comparison_path.exists()
    else {}
)


def dense_save(name, model, prediction, seconds):
    metrics, levels = price_evaluate(dense_evaluator, dense_actual, prediction)

    directory = dense_results / name
    directory.mkdir(parents=True, exist_ok=True)

    if name == "CatBoost":
        model.save_model(str(dense_models / "catboost.cbm"))
    else:
        model.booster_.save_model(
            str(dense_models / "lightgbm.txt"),
            num_iteration=386,
        )

    prediction.to_csv(
        directory / "predictions.csv.gz",
        compression="gzip",
    )
    levels.to_csv(directory / "metrics_by_level.csv")

    (directory / "metrics.json").write_text(
        json.dumps(
            {**metrics, "seconds": seconds},
            indent=2,
        ),
        encoding="utf-8",
    )

    control = dense_controls[name]
    dense_rows[name] = {
        "WRMSSE_52x2": control["WRMSSE"],
        "WRMSSE_52x4": metrics["WRMSSE"],
        "gain": 1 - metrics["WRMSSE"] / control["WRMSSE"],
        "Bias_52x2": control["Bias"],
        "Bias_52x4": metrics["Bias"],
        "minutes": seconds / 60,
    }

    table = pd.DataFrame.from_dict(dense_rows, orient="index")
    table.to_csv(dense_comparison_path)

    display(
        table.style.format(
            {
                "WRMSSE_52x2": "{:.5f}",
                "WRMSSE_52x4": "{:.5f}",
                "gain": "{:+.2%}",
                "Bias_52x2": "{:+.2%}",
                "Bias_52x4": "{:+.2%}",
                "minutes": "{:.1f}",
            }
        )
    )


def dense_checkpoint():
    archive = Path("/kaggle/working/demand-planner_checkpoint.zip")
    temporary = archive.with_name("demand-planner_checkpoint.tmp.zip")

    with ZipFile(temporary, "w", ZIP_DEFLATED, compresslevel=1) as output:
        for folder in (
            dense_root / "models",
            dense_root / "results",
        ):
            for path in sorted(folder.rglob("*")):
                if path.is_file():
                    output.write(
                        path,
                        path.relative_to(dense_root),
                    )

    temporary.replace(archive)
    print("Архив моделей и результатов:", archive)
    display(FileLink(relpath(archive, Path.cwd())))


def dense_build_train():
    feature_parts = []
    target_parts = []

    for number, (origin, horizons) in enumerate(dense_plan, start=1):
        for horizon in horizons:
            target_date = origin + pd.Timedelta(days=horizon)
            position = daily_history_dates.get_indexer([target_date])[0]

            if position < 0 or target_date > dense_origin:
                raise ValueError(f"Недоступная целевая дата: {target_date}")

            part = make_daily_features_with_cross_store(origin, target_date)

            if not part.index.equals(dense_evaluator.series_index):
                raise ValueError("Нарушен порядок товаров и магазинов.")

            feature_parts.append(part.loc[:, dense_columns].reset_index(drop=True))
            target_parts.append(daily_history_values[:, position].astype("float32"))

        if number % 13 == 0:
            print(
                f"Подготовлено дат: {number}/52",
                flush=True,
            )

    X = pd.concat(feature_parts, ignore_index=True)
    y = np.concatenate(target_parts)

    if len(X) != 6_341_920 or len(y) != len(X):
        raise ValueError("Ожидалось 6 341 920 обучающих строк.")

    return lg_prepare_categories(X), y


dense_data_started = perf_counter()

dense_X, dense_y = dense_build_train()
dense_X_valid = lg_forecast_data(dense_origin, dense_evaluator)
dense_actual = daily_history.reindex(
    index=dense_evaluator.series_index,
    columns=dense_evaluator.forecast_dates,
)

print(f"Обучающих строк: {len(dense_X):,}; " f"признаков: {dense_X.shape[1]}")
print(f"Подготовка: " f"{(perf_counter() - dense_data_started) / 60:.1f} мин.")
print("Далее два обучения: CatBoost 768 деревьев, " "LightGBM 386 деревьев.")


Подготовлено дат: 13/52
Подготовлено дат: 26/52
Подготовлено дат: 39/52
Подготовлено дат: 52/52
Обучающих строк: 6,341,920; признаков: 63
Подготовка: 0.2 мин.
Далее два обучения: CatBoost 768 деревьев, LightGBM 386 деревьев.


### 7.1. CatBoost 52 × 4

WRMSSE вырос с **0.56683 до 0.57435**: ухудшение **1.33%**. Bias изменился с **−1.66% до −1.87%**. Обучение, прогноз и сохранение заняли **40.6 минуты**. Рабочим вариантом пока остаётся 52 × 2.


In [50]:
print("Обучение 1/2: CatBoost, 52 × 4.", flush=True)

dense_cb_started = perf_counter()

dense_cb_model = CatBoostRegressor(
    **dense_cb_params,
    verbose=128,
)
dense_cb_model.fit(
    dense_X,
    dense_y,
    cat_features=dense_categories,
    use_best_model=False,
)

dense_cb_values = dense_cb_model.predict(
    dense_X_valid,
    prediction_type="Exponent",
)

if not np.isfinite(dense_cb_values).all() or (dense_cb_values < 0).any():
    raise ValueError("Некорректные прогнозы CatBoost.")

dense_cb_prediction = pd.DataFrame(
    dense_cb_values.reshape(28, len(dense_evaluator.series_index)).T,
    index=dense_evaluator.series_index,
    columns=dense_evaluator.forecast_dates,
)

dense_save(
    "CatBoost",
    dense_cb_model,
    dense_cb_prediction,
    perf_counter() - dense_cb_started,
)

del dense_cb_model, dense_cb_values
gc.collect()

dense_checkpoint()


Обучение 1/2: CatBoost, 52 × 4.
0:	learn: 4.5116651	total: 3.94s	remaining: 50m 20s
128:	learn: 3.6589155	total: 6m 27s	remaining: 31m 57s
256:	learn: 3.6289425	total: 13m 1s	remaining: 25m 54s
384:	learn: 3.6127085	total: 19m 47s	remaining: 19m 41s
512:	learn: 3.6025575	total: 26m 37s	remaining: 13m 13s
640:	learn: 3.5949709	total: 33m 24s	remaining: 6m 37s
767:	learn: 3.5897662	total: 40m 9s	remaining: 0us


,WRMSSE_52x2,WRMSSE_52x4,gain,Bias_52x2,Bias_52x4,minutes
CatBoost,0.56683,0.57435,-1.33%,-1.66%,-1.87%,40.6


Архив моделей и результатов: /kaggle/working/demand-planner_checkpoint.zip


/kaggle/working/demand-planner_checkpoint.zip

### 7.2. LightGBM 52 × 4

WRMSSE снизился с **0.55498 до 0.55332**: улучшение **0.30%**. Bias изменился с **−2.20% до −2.03%**. Обучение, прогноз и сохранение заняли **6.3 минуты**.

Плюс небольшой и пока получен на одном окне. Скорость позволяет следующим опытом проверить 52 × 8. После выполнения скачиваем обновлённый checkpoint и отдельно сохраняем ноутбук.


In [51]:
print("Обучение 2/2: LightGBM, 52 × 4.", flush=True)

dense_lg_started = perf_counter()


def dense_lg_progress(env):
    iteration = env.iteration + 1

    if iteration % 100 == 0 or iteration == 386:
        minutes = (perf_counter() - dense_lg_started) / 60
        print(
            f"LightGBM: {iteration}/386 деревьев, " f"{minutes:.1f} мин.",
            flush=True,
        )


dense_lg_model = lgb.LGBMRegressor(**dense_lg_params)
dense_lg_model.fit(
    dense_X,
    dense_y,
    categorical_feature=dense_categories,
    callbacks=[dense_lg_progress],
)

dense_lg_values = dense_lg_model.predict(dense_X_valid)

if not np.isfinite(dense_lg_values).all() or (dense_lg_values < 0).any():
    raise ValueError("Некорректные прогнозы LightGBM.")

dense_lg_prediction = pd.DataFrame(
    dense_lg_values.reshape(28, len(dense_evaluator.series_index)).T,
    index=dense_evaluator.series_index,
    columns=dense_evaluator.forecast_dates,
)

dense_save(
    "LightGBM",
    dense_lg_model,
    dense_lg_prediction,
    perf_counter() - dense_lg_started,
)

del (
    dense_lg_model,
    dense_lg_values,
    dense_X,
    dense_y,
    dense_X_valid,
)
gc.collect()

dense_checkpoint()

print(
    "Готово. Положительный gain означает улучшение "
    "относительно своего контроля 52 × 2."
)
print("Результаты:", dense_results)


Обучение 2/2: LightGBM, 52 × 4.
LightGBM: 100/386 деревьев, 1.9 мин.
LightGBM: 200/386 деревьев, 3.3 мин.
LightGBM: 300/386 деревьев, 4.6 мин.
LightGBM: 386/386 деревьев, 5.8 мин.


,WRMSSE_52x2,WRMSSE_52x4,gain,Bias_52x2,Bias_52x4,minutes
CatBoost,0.56683,0.57435,-1.33%,-1.66%,-1.87%,40.6
LightGBM,0.55498,0.55332,+0.30%,-2.20%,-2.03%,6.3


Архив моделей и результатов: /kaggle/working/demand-planner_checkpoint.zip


/kaggle/working/demand-planner_checkpoint.zip

Готово. Положительный gain означает улучшение относительно своего контроля 52 × 2.
Результаты: /kaggle/working/demand-planner/results/m5_wrmsse/density_52x4_v1/2016-03-27


## 8. Итоги на 22 октября 2026 года

### Историческое окно: 28 марта — 24 апреля 2016 года

| Модель | Схема | WRMSSE ↓ | Bias | Изменение WRMSSE к своей 52 × 2 |
|---|---|---:|---:|---:|
| CatBoost, локальный контроль | 52 × 2 | 0.56683 | −1.66% | — |
| CatBoost, Kaggle | 52 × 4 | 0.57435 | −1.87% | −1.33% |
| LightGBM, Kaggle | 52 × 2 | 0.55498 | −2.20% | — |
| LightGBM, Kaggle | 52 × 4 | 0.55332 | −2.03% | +0.30% |

Для CatBoost увеличение плотности пока не помогло. Контроль взят локальный, поэтому точный эффект между средами не изолирован. Для LightGBM небольшой плюс получен на одном окне и ещё требует проверки на других периодах.

### Public: 25 апреля — 22 мая 2016 года

| Модель | Схема | WRMSSE ↓ | Bias |
|---|---|---:|---:|
| CatBoost, Kaggle до исправления цены | 52 × 2 | 0.63994 | −4.31% |
| LightGBM, Kaggle | 52 × 2 | 0.67879 | −5.98% |

Более плотные модели на public ещё не проверяли. Лучший исторический результат LightGBM не перенёсся в преимущество на public.

## 9. С чего продолжить

1. Подключить сохранённые модели и прогнозы из checkpoint. Подготовительные функции нужны отдельно: переменные завершённой сессии не входят в архив.
2. Проверить **LightGBM 52 × 8**: те же 63 признака и 386 деревьев, сравнение с 52 × 2 и 52 × 4. Записать время и расход памяти. Для CatBoost пока сохранить 52 × 2.
3. Подтвердить удачный вариант на других исторических окнах. Затем проверить новые признаки и настройки LightGBM на выбранной плотности.
4. Получить прогнозы лучших вариантов на одинаковых окнах, выбрать веса ансамбля и коэффициент калибровки по WRMSSE.
5. Зафиксировать решение, подготовить private-прогноз и submission. Перед отправкой проверить доступность Late Submission.

Архив `demand-planner_checkpoint.zip` содержит `models/` и `results/`, включая предыдущие эксперименты. Сам `.ipynb` сохраняется отдельно. В Git входят ноутбук, небольшие таблицы и JSON-настройки; модели и полные прогнозы исключены.
